# InsureAssist — LoRA Fine-Tuning (Phase 2)

**Goal:** teach a small open LLM to answer insurance questions in our style, cheaply.

**Key idea — LoRA (Low-Rank Adaptation):** training a full LLM needs many expensive GPUs.
LoRA *freezes* the big model and trains only a tiny set of extra weights ("adapter").
Result: fine-tuning runs on a **free Colab T4 GPU** in minutes, and the adapter file is
only a few MB.

**Run this on Google Colab:** File -> Upload notebook -> pick this file. Then
Runtime -> Change runtime type -> **T4 GPU** -> Save. Run cells top to bottom.

What you'll learn: Hugging Face `transformers`, `peft` (LoRA), `trl` (SFTTrainer),
and experiment tracking with **MLflow**.

## 1. Install libraries

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" "trl>=0.9" "datasets>=2.20" \
    "accelerate>=0.33" "bitsandbytes>=0.43" mlflow

## 2. Check the GPU
If this errors or shows no GPU, set Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
!nvidia-smi -L

## 3. Training data
We fine-tune on instruction/answer pairs about insurance. In a real project you'd have
thousands; here a small set shows the full workflow. Each example is a chat conversation.

In [ ]:
from datasets import Dataset

raw = [
    {"q": "Does home insurance cover water damage from a burst pipe?",
     "a": "Yes. Sudden and accidental water damage from a burst pipe is covered, including tracing and accessing the leak. Gradual leakage or wear and tear is not covered."},
    {"q": "What is the excess for an escape of water claim?",
     "a": "A higher excess of EUR 500 applies to escape-of-water claims, versus the standard EUR 250."},
    {"q": "Up to how much is jewellery covered per item?",
     "a": "Jewellery is covered up to a single-item limit of EUR 2,000 unless separately listed on the schedule."},
    {"q": "How long do I have to report a home claim?",
     "a": "Home insurance claims must be reported within 30 days of the incident."},
    {"q": "Does comprehensive auto cover include a courtesy car?",
     "a": "Yes, up to 14 days while your vehicle is repaired by an approved garage."},
    {"q": "What extra excess applies to drivers under 25?",
     "a": "An additional young-driver excess of EUR 300 on top of the EUR 400 own-damage excess."},
    {"q": "What is the maximum no-claims discount?",
     "a": "Up to 65%, reached after five claim-free years."},
    {"q": "Is mechanical breakdown covered by auto insurance?",
     "a": "No, mechanical or electrical breakdown is excluded."},
    {"q": "Is damage covered if the home is empty for two months?",
     "a": "No. Damage is excluded if the home is unoccupied for more than 60 consecutive days."},
    {"q": "What is needed before a car theft claim is processed?",
     "a": "A police report reference number is required."},
]

def to_chat(ex):
    return {"messages": [
        {"role": "system", "content": "You are a precise insurance policy assistant."},
        {"role": "user", "content": ex["q"]},
        {"role": "assistant", "content": ex["a"]},
    ]}

ds = Dataset.from_list([to_chat(r) for r in raw])
print(ds)
print(ds[0])

## 4. Load the base model (4-bit) + tokenizer
We load **Phi-3-mini** in 4-bit to fit the free GPU. 4-bit = weights stored compactly.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto", trust_remote_code=True
)
print("Base model loaded.")

## 5. Attach the LoRA adapter
`r` = adapter size (bigger = more capacity, slower). `target_modules` = which layers get
the adapter (the attention projections). Only these tiny weights will be trained.

In [ ]:
from peft import LoraConfig

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

## 6. Train with TRL's SFTTrainer + log to MLflow
SFT = Supervised Fine-Tuning. MLflow records params, loss, and saves the adapter so you
can compare runs later (this is the 'experiment tracking' skill).

In [ ]:
import mlflow
from trl import SFTTrainer, SFTConfig

mlflow.set_experiment("insureassist-lora")

args = SFTConfig(
    output_dir="adapter",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    bf16=True,
    report_to=[],           # we log to MLflow manually below
    max_seq_length=1024,
)

with mlflow.start_run():
    mlflow.log_params({
        "base_model": BASE_MODEL, "r": lora.r, "lora_alpha": lora.lora_alpha,
        "epochs": args.num_train_epochs, "lr": args.learning_rate,
    })
    trainer = SFTTrainer(model=model, args=args, train_dataset=ds, peft_config=lora)
    trainer.train()
    trainer.save_model("adapter")          # saves ONLY the small LoRA adapter
    mlflow.log_artifacts("adapter", artifact_path="lora_adapter")
    print("Training done. Adapter saved to ./adapter")

## 7. Quick test — does it answer in our style?

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer)
msg = [
    {"role": "system", "content": "You are a precise insurance policy assistant."},
    {"role": "user", "content": "Is a burst pipe covered by home insurance?"},
]
out = pipe(msg, max_new_tokens=120, do_sample=False)
print(out[0]["generated_text"][-1]["content"])

## 8. Download the adapter
Download `adapter/` (right-click in the Colab file browser -> Download) OR push to the
Hugging Face Hub. Put it in your repo at `finetune/adapter/` for Phase 3.

```python
# Optional: push to Hugging Face Hub
# from huggingface_hub import login; login()
# trainer.model.push_to_hub("mzquadri/insureassist-phi3-lora")
```

**Next:** Phase 3 — plug this adapter into the RAG pipeline and evaluate with RAGAS.